# CatKG: literature extraction and tandem pathway recommendation
This notebook follows the article inputs, Gemini 2.5 Flash extraction prompts, saved extraction results, cleaning, graph construction, and pathway ranking. The sample comes from the paper's cleaned 471,436-reaction Neo4j graph. Run from the package directory using the environment described in README.md. Default cells run offline and use saved extraction results; live Gemini extraction and Neo4j retrieval are optional.

In [ ]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode = True
ROOT = Path.cwd().resolve()
if not (ROOT / 'process_reactions_and_rank_paths.py').exists():
    raise RuntimeError('Open this notebook with the code-package directory as the working directory.')
sys.path.insert(0, str(ROOT))
from data_processing.clean_reactions import read
from process_reactions_and_rank_paths import run
OUTPUT = ROOT / 'outputs'
config = read(ROOT / 'extraction/model_settings.json')
assert config['model'] == 'gemini-2.5-flash'
print('Extraction model:', config['model'])


## 1. Forty full-text inputs and publication correspondence


In [ ]:
from extraction.extract_reactions import validate
validation = validate(ROOT, ROOT / 'data/fulltexts', OUTPUT)
assert validation['articles'] == 40 and not validation['live_model_called']
article = read(ROOT / 'data/fulltexts/photocatalysis_01.json')
print('Validated article inputs:', validation['articles'])
print('Example DOI:', article['doi'])
print('Body paragraphs:', len(article['paragraphs']))


## 2. Prompts and extraction stages
`extraction/prompts.json` contains classification, coarse extraction, and refinement prompts, together with detail extraction and refinement prompts for each catalytic mode. The following cell formats an article and prepares a request for inspection.


In [ ]:
from extraction.extract_reactions import format_article_content, build_request
prompts = read(ROOT / 'extraction/prompts.json')
print('Prompt/template names:', list(prompts))
print('Photocatalysis prompt preview:', prompts['photo_detail_extraction_model'][:500])
request = build_request(prompts['coarse_extraction_model'], '"""\n' + format_article_content(article) + '\n"""', config)
print('Formatted article characters:', len(format_article_content(article)))
print('API request sent:', False)


## 3. Clean the saved extraction results
The example retains raw performance values and units, normalizes supported percentage expressions, and lists unit-bearing values with their cleaned counterparts for review.


In [ ]:
summary = run(ROOT, OUTPUT)
assert summary['sample_articles'] == 40 and summary['source_graph_reactions'] == 471436
print(json.dumps(summary, indent=2))
print('Raw extraction articles:', len(read(ROOT / 'data/extracted_reactions.json')))
print('Unit-bearing values flagged:', len(read(OUTPUT / 'performance_review.json')))


## 4. Prepare the graph data
The offline workflow generates node and relationship tables in `outputs/graph/`; it does not write to a Neo4j database. Shared compounds connect reaction steps, and catalyst and publication relationships link each reaction to its catalyst, article, journal, and publisher.

Install Neo4j Community 5.20.0 and Java 17 separately, then choose either of the following approaches:

**Option 1: Restore the supplied dump.** Load `data/neo4j/neo4j.dump` into a stopped, empty Neo4j instance using `neo4j-admin database load`, with `data/neo4j/` as the input directory. Then start the instance. The dump already contains the sample nodes, relationships, and properties, so rebuilding the graph from the cleaned records is unnecessary. See the [dump restoration commands](data_processing/README.md#restore-the-graph).

**Option 2: Build from the cleaned records.** Run the preceding cells to generate `outputs/graph/`, start a dedicated empty database, and set the connection variables for that instance. From a terminal in the repository directory, run:

```shell
python data_processing/neo4j_graph.py --import-demo
```

The script creates uniqueness constraints and imports the generated nodes and relationships, then retrieves the sample paths and checks their scores against the offline results. See the [graph import instructions](data_processing/README.md#build-a-graph-from-the-cleaned-records).

For either approach, set `NEO4J_URI`, `NEO4J_DATABASE`, and `NEO4J_USERNAME` to match the running instance; the Python tools prompt for the password if `NEO4J_PASSWORD` is unset. Section 8 can then retrieve and score paths from the graph. Connection settings are described in the [Neo4j instructions](data_processing/README.md#neo4j-demonstration).


In [ ]:
print(json.dumps(summary['graph_counts'], indent=2))
print('Example reaction-to-article correspondence:')
import csv
with (ROOT / 'data/samples.csv').open(encoding='utf-8', newline='') as handle:
    sample = next(csv.DictReader(handle))
print(json.dumps(sample, indent=2))


## 5. Retrieve and rank two-step paths
CO connects the photocatalytic and thermocatalytic reactions. Scores are recalculated for all 400 pairs in the sample.


In [ ]:
paths = read(OUTPUT / 'path_rankings.json')
assert len(paths) == 400
assert [p['total'] for p in paths] == sorted([p['total'] for p in paths], reverse=True)
for p in paths[:3]:
    print(p['rank'], p['photo_ID'], '->', p['thermal_ID'], p['total'])


## 6. Scoring and ranking example
See `path_recommendation/scoring_and_ranking_example.py` and `path_recommendation/README.md` for the corresponding explanation. Run `python path_recommendation/scoring_and_ranking_example.py` to print each score contribution for the highest-ranked path.


In [ ]:
example = paths[0]
print(json.dumps(example, indent=2))
for row in read(OUTPUT / 'reaction_score_breakdown.json'):
    if row['source_reaction_ID'] in {example['photo_ID'], example['thermal_ID']}:
        print(json.dumps(row, indent=2))
from path_recommendation.scoring import pair_breakdown
print('Temperature contribution for 973 K and 298 K:',
      pair_breakdown({'temperature': '973 K'}, {'temperature': '298 K'})['terms']['B5'])


## 7. Optional new Gemini extraction
Set `RUN_LIVE_EXTRACTION = True` and supply `GOOGLE_API_KEY` to send this article to Gemini. The default run uses the saved extraction results.


In [ ]:
RUN_LIVE_EXTRACTION = False
if RUN_LIVE_EXTRACTION:
    from extraction.extract_reactions import extract_article
    from data_processing.clean_reactions import write
    extracted, audit = extract_article(article)
    write(ROOT / 'outputs_live/photocatalysis_01_extraction.json', extracted)
    write(ROOT / 'outputs_live/photocatalysis_01_extraction.audit.json', audit)
else:
    print('Live Gemini extraction skipped; no API call made.')


## 8. Optional Neo4j retrieval
Set the connection variables described in `data_processing/README.md` and enable the following cell to retrieve the sample reactions and connected paths from Neo4j, recalculate their scores, and print the highest-ranked path. The same calculation is available as `path_recommendation/scoring_and_ranking_example.py --source neo4j`.


In [ ]:
RUN_NEO4J = False
if RUN_NEO4J:
    import getpass
    import os
    from neo4j import GraphDatabase, READ_ACCESS
    from path_recommendation.recommendation import compound_names, retrieve_and_rank, sample_reaction_ids
    from path_recommendation.scoring_and_ranking_example import show_ranked_paths
    password = os.getenv('NEO4J_PASSWORD') or getpass.getpass('Neo4j password: ')
    auth = (os.getenv('NEO4J_USERNAME', 'neo4j'), password)
    with GraphDatabase.driver(os.getenv('NEO4J_URI', 'bolt://localhost:7687'), auth=auth) as driver:
        with driver.session(database=os.getenv('NEO4J_DATABASE', 'neo4j'), default_access_mode=READ_ACCESS) as session:
            ranked, records, summary = session.execute_read(retrieve_and_rank, sample_reaction_ids(ROOT), compound_names(ROOT))
    print(json.dumps(summary, indent=2))
    show_ranked_paths(ranked, records, 'the sample paths retrieved from Neo4j')
else:
    print('Neo4j query skipped. Set RUN_NEO4J = True after starting the database.')
